# Imports

In [ ]:
import pandas as pd
import numpy as np
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
export_dir = os.getcwd()
from pathlib import Path
import pickle
from collections import defaultdict
import time
import torch
import torch.nn as nn
import copy
import torch.nn.functional as F
import optuna
import logging
import matplotlib.pyplot as plt
import random
import wandb
import ipynb
import importlib

In [ ]:
data_name = "ML1M" ### Can be ML1M, Yahoo, Pinterest
recommender_name = "MLP" ## Can be MLP, VAE
DP_DIR = Path("processed_data", data_name) 
#export_dir = Path(os.getcwd())
export_dir = Path("/home/mvarasteh/environments/CLXR-TRSL/CLXR/code")
files_path = Path(export_dir.parent, DP_DIR)
checkpoints_path = Path(export_dir, "checkpoints")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
output_type_dict = {
    "VAE":"multiple",
    "MLP":"single"
}

num_users_dict = {
    "ML1M":6037,
    "Yahoo":13797, 
    "Pinterest":19155
}

num_items_dict = {
    "ML1M":3381,
    "Yahoo":4604, 
    "Pinterest":9362
}



recommender_path_dict = {
    #("ML1M","VAE"): Path(checkpoints_path, "VAE_ML1M_0.0007_128_10.pt"),
    ("ML1M","VAE"): Path(checkpoints_path, "VAE_ML1M_0.0003_64.pt"),
    ("ML1M","MLP"):Path(checkpoints_path, "MLP1_ML1M_0.0076_256_7.pt"),
    
    #("Yahoo","VAE"): Path(checkpoints_path, "VAE_Yahoo_0.0001_128_13.pt"),
    ("Yahoo","VAE"): Path(checkpoints_path, "VAE_Yahoo_128.pt"),

    ("Yahoo","MLP"):Path(checkpoints_path, "MLP2_Yahoo_0.0083_128_1.pt"),

    ("Pinterest","VAE"): Path(checkpoints_path, "VAE_Pinterest_0.0002_32_12.pt"),
    ("Pinterest","MLP"):Path(checkpoints_path, "MLP_Pinterest_0.0062_512_21_0.pt")
    
}


hidden_dim_dict = {
    #("ML1M","VAE"): None,
    ("ML1M","VAE"): [256,64],
    ("ML1M","MLP"): 32,

    ("Yahoo","VAE"): None,
    ("Yahoo","MLP"):32,
    
    ("Pinterest","VAE"): None,
    ("Pinterest","MLP"):512

}

In [ ]:
output_type = output_type_dict[recommender_name] ### Can be single, multiple
num_users = num_users_dict[data_name] 
num_items = num_items_dict[data_name] 
num_features = num_items_dict[data_name]
hidden_dim = hidden_dim_dict[(data_name,recommender_name)]

recommender_path = recommender_path_dict[(data_name,recommender_name)]

## Data imports and preprocessing

In [ ]:
train_data = pd.read_csv(Path(files_path,f'train_data_{data_name}.csv'), index_col=0)
test_data = pd.read_csv(Path(files_path,f'test_data_{data_name}.csv'), index_col=0)
train_data['user_id'] = train_data.index
test_data['user_id'] = test_data.index
static_test_data = pd.read_csv(Path(files_path,f'static_test_data_{data_name}.csv'), index_col=0)
with open(Path(files_path,f'pop_dict_{data_name}.pkl'), 'rb') as f:
    pop_dict = pickle.load(f)
train_array = train_data.to_numpy()
test_array = test_data.to_numpy()
items_array = np.eye(num_items)
all_items_tensor = torch.Tensor(items_array).to(device)

In [ ]:
pop_array = np.zeros(len(pop_dict))
for key, value in pop_dict.items():
    pop_array[key] = value

In [ ]:
kw_dict = {'device':device,
          'num_items': num_items,
          'pop_array':pop_array,
          'all_items_tensor':all_items_tensor,
          'static_test_data':static_test_data,
          'items_array':items_array,
          'output_type':output_type,
          'recommender_name':recommender_name}

# Recommenders Architecture

In [ ]:
from ipynb.fs.defs.recommenders_architecture import *
importlib.reload(ipynb.fs.defs.recommenders_architecture)
from ipynb.fs.defs.recommenders_architecture import *

In [ ]:
VAE_config = { "enc_dims": [256, 64], "dropout": 0.5, "anneal_cap": 0.2, "total_anneal_steps": 200000 }


In [ ]:
def load_recommender():
    if recommender_name=='MLP':
        recommender = MLP(hidden_dim, **kw_dict)
    elif recommender_name=='VAE':
        recommender = VAE(VAE_config, **kw_dict)
    elif recommender_name=='MLP_model':
        recommender = MLP_model(hidden_size=hidden_dim, num_layers=3, **kw_dict)
    elif recommender_name=='GMF_model':
        recommender = GMF_model(hidden_size=hidden_dim, **kw_dict)
    elif recommender_name=='NCF':
        MLP_temp = MLP_model(hidden_size=hidden_dim, num_layers=3, **kw_dict)
        GMF_temp = GMF_model(hidden_size=hidden_dim, **kw_dict)
        recommender = NCF(factor_num=hidden_dim, num_layers=3, dropout=0.5, model= 'NeuMF-pre', GMF_model= GMF_temp, MLP_model=MLP_temp, **kw_dict)
    recommender_checkpoint = torch.load(Path(checkpoints_path, recommender_path), map_location=torch.device('cpu'))
    recommender.load_state_dict(recommender_checkpoint)
    recommender.eval()
    for param in recommender.parameters():
        param.requires_grad= False
    return recommender
    
recommender = load_recommender()

# Help functions

In [ ]:
from ipynb.fs.defs.help_functions import *
importlib.reload(ipynb.fs.defs.help_functions)
from ipynb.fs.defs.help_functions import *

In [ ]:
def get_user_recommended_item(user_tensor, recommender, **kw):
    all_items_tensor = kw['all_items_tensor']
    num_items = kw['num_items']
    user_res = recommender_run(user_tensor, recommender, all_items_tensor, None, 'vector', **kw)[:num_items]
    user_tensor = user_tensor[:num_items]
    user_catalog = torch.ones_like(user_tensor)-user_tensor
    user_recommenations = torch.mul(user_res, user_catalog)
    # Get the indices of the items sorted by their recommendation score in descending order
    sorted_recommendations = torch.argsort(user_recommenations, descending=True)
    return(sorted_recommendations[0:10])

## Load / create top recommended items dict

In [ ]:
## Load / create top recommended items dict

create_dicts = True
if create_dicts:

    ## target and contrastive items for training 
    top1_train, top2_train = {}, {}

    ## target and contrastive items for testing 
    top1_test, top2_test = {}, {}

    ##recommendations for each user in training dataset
    recom_train,recom_test= {}, {}

    for i in range(train_array.shape[0]):
        user_index = train_array[i][-1]
        user_tensor = torch.Tensor(train_array[i][:-1]).to(device)
        recomm_list=get_user_recommended_item(user_tensor, recommender, **kw_dict)

        top1_train[user_index] = int(recomm_list[0])
        ## Sampling for the contrastive item below the target item
        top2_train[user_index]= np.array(recomm_list[1:9].cpu())

    for i in range(test_array.shape[0]):
        user_index = test_array[i][-1]
        user_tensor = torch.Tensor(test_array[i][:-1]).to(device)
        recomm_list=get_user_recommended_item(user_tensor, recommender, **kw_dict)
        ## the target item is the first rank item and contrastive item is rank 7+1=8
        top1_test[user_index] = int(recomm_list[0])
        top2_test[user_index] = np.array(recomm_list[1:9].cpu())
        
    with open(Path(files_path,f'top1_train_{data_name}_{recommender_name}.pkl'), 'wb') as f:
        pickle.dump(top1_train, f)
    with open(Path(files_path,f'top2_train_{data_name}_{recommender_name}.pkl'), 'wb') as f:
        pickle.dump(top2_train, f)
    with open(Path(files_path,f'top1_test_{data_name}_{recommender_name}.pkl'), 'wb') as f:
        pickle.dump(top1_test, f)
    with open(Path(files_path,f'top2_test_{data_name}_{recommender_name}.pkl'), 'wb') as f:
        pickle.dump(top2_test, f)
else:
    with open(Path(files_path,f'top1_train_{data_name}_{recommender_name}.pkl'), 'rb') as f:
        top1_train = pickle.load(f)
    with open(Path(files_path,f'top2_train_{data_name}_{recommender_name}.pkl'), 'rb') as f:
        top2_train = pickle.load(f)
    with open(Path(files_path,f'top1_test_{data_name}_{recommender_name}.pkl'), 'rb') as f:
        top1_test = pickle.load(f)
    with open(Path(files_path,f'top2_test_{data_name}_{recommender_name}.pkl'), 'rb') as f:
        top2_test = pickle.load(f)

# LXR Architecture

In [ ]:
class Explainer(nn.Module):
    def __init__(self, user_size, item_size, hidden_size):
        super(Explainer, self).__init__()
        self.users_fc = nn.Linear(in_features=user_size, out_features=hidden_size)
        self.items_fc = nn.Linear(in_features=item_size, out_features=hidden_size)
        self.items_comp_fc = nn.Linear(in_features=item_size, out_features=hidden_size)
        self.bottleneck = nn.Sequential(
            nn.Tanh(),
            nn.Linear(in_features=hidden_size * 3, out_features=hidden_size * 2),
            nn.Tanh(),
            nn.Linear(in_features=hidden_size * 2, out_features=hidden_size),
            nn.Tanh(),
            nn.Linear(in_features=hidden_size, out_features=user_size),
            nn.Sigmoid()
        )
        
    def forward(self, user_tensor, item_tensor, item_comp_tensor):
        user_output = self.users_fc(user_tensor.float())
        item_output = self.items_fc(item_tensor.float())
        item_output_comp = self.items_comp_fc(item_comp_tensor.float())
        combined_output = torch.cat((user_output, item_output, item_output_comp), dim=-1)
        expl_scores = self.bottleneck(combined_output)
        return expl_scores

# Train functions

In [ ]:
class LXR_loss(nn.Module):
    def __init__(self, lambda_pos, lambda_neg, alpha, lambda_dlr, k):
        super(LXR_loss, self).__init__()
        
        self.lambda_pos = lambda_pos
        self.lambda_neg = lambda_neg
        self.alpha = alpha
        self.lambda_dlr = lambda_dlr
        self.k = k



    def forward(self, user_tensors, items_tensors, items_ids, pos_masks, items_comp):
        # Create positive and negative masks
        neg_masks = torch.sub(torch.ones_like(pos_masks), pos_masks)
        x_masked_pos = user_tensors * pos_masks
        x_masked_neg = user_tensors * neg_masks
        
        # Compute logits for positive and negative samples
        if output_type == 'single':
            x_masked_res_pos = torch.diag(recommender_run(x_masked_pos, recommender, items_tensors, item_id=items_ids, wanted_output = 'single', **kw_dict))
            
            comp_masked_pos = torch.diag(recommender_run(x_masked_pos, recommender, torch.Tensor(items_array[items_comp]).to(device), item_id=items_comp, wanted_output = 'single', **kw_dict))
           
            #items_last_k_pos = torch.diag(recommender_run(x_masked_pos, recommender, torch.Tensor(items_array[items_last_k]).to(device), item_id=items_last_k, wanted_output = 'single', **kw_dict))
            
            x_masked_res_neg = torch.diag(recommender_run(x_masked_neg, recommender, items_tensors, item_id=items_ids, wanted_output = 'single', **kw_dict))

        else:
            x_masked_res_pos_before = recommender_run(x_masked_pos, recommender, items_tensors,
                                                                  item_id=items_ids,
                                                                  wanted_output='vector', **kw_dict)
            comp_masked_pos_before = recommender_run(x_masked_pos, recommender, torch.Tensor(items_array[items_comp]).to(device),
                                                                  item_id=items_comp,
                                                                  wanted_output='vector', **kw_dict)
            
            #items_last_k_pos_before = recommender_run(x_masked_pos, recommender, torch.Tensor(items_array[items_last_k]).to(device),
                                                                  #item_id=items_last_k,
                                                                  #wanted_output='vector', **kw_dict)

            x_masked_res_neg_before = recommender_run(x_masked_neg, recommender, items_tensors,
                                                                  item_id=items_ids,
                                                                  wanted_output='vector', **kw_dict)
            rows = torch.arange(len(items_ids))
            x_masked_res_pos = x_masked_res_pos_before[rows, items_ids]
            comp_masked_pos = comp_masked_pos_before[rows, items_ids]
            #items_last_k_pos = items_last_k_pos_before[rows, items_ids]
            
            x_masked_res_neg = x_masked_res_neg_before[rows, items_ids]
        
        
        # Compute main loss components
        epsilon = 1e-8
        
        pos_loss = -torch.mean(torch.log(x_masked_res_pos + epsilon))
        neg_loss = torch.mean(torch.log(x_masked_res_neg + epsilon))
        l1 = x_masked_pos[user_tensors>0].mean()

        
        numerator = torch.sub(x_masked_res_pos,comp_masked_pos) 
        #denominator = x_masked_res_pos + items_last_k_pos   # normalization term
        tdlr_loss = -numerator#/(denominator + epsilon)  # Avoid division by zero by adding epsilon
        tdlr_avg = tdlr_loss.mean()  # Average loss over the batch

    
        # Combine all losses 
        combined_loss = self.lambda_pos * pos_loss + self.lambda_neg * neg_loss + self.alpha * l1 + self.lambda_dlr * tdlr_avg
        
        return combined_loss, pos_loss, neg_loss, l1

In [ ]:
#LXR based similarity
def find_LXR_mask(user_tensor, item_id, item_tensor, item_comp_tensor, explainer):
    expl_scores = explainer(user_tensor, item_tensor, item_comp_tensor)
    x_masked = user_tensor*expl_scores
    item_sim_dict = {i: x_masked[i].item() for i in range(len(x_masked))}    

    return item_sim_dict

In [ ]:
def calculate_pos_neg_k(user_tensor, first_item_id, second_item_id, first_items_tensor, second_items_tensor, num_of_bins, explainer, k):
    
    user_hist_size = int(torch.sum(user_tensor))
    bins = [0] + [len(x) for x in np.array_split(np.arange(user_hist_size), num_of_bins, axis=0)]

    results = {
        "res1": None, "res2": None
    }

    # Helper function to mask items
    def mask_items(user_tensor, sim_items, total_items, device):
        mask = torch.zeros_like(user_tensor, dtype=torch.float32, device=device)
        indices = [item[0] for item in sim_items[:total_items]]
        mask[indices] = 1
        return user_tensor - mask

    # Process each item set (first, second, comparative)
    def process_sim_items(comp_sim_items, first_item_id, second_item_id,bins, user_tensor, user_hist_size, recommender, **kw_dict):
      
        comp_sorted_sim_items = list(sorted(comp_sim_items.items(), key=lambda item: item[1], reverse=True))[:user_hist_size]
        reverse_comp_sim_items = list(sorted(comp_sim_items.items(), key=lambda item: item[1]))[:user_hist_size]

        total_items = 0
        for i, bin_size in enumerate(bins):
            total_items += bin_size
            
            comp_POS_masked = mask_items(user_tensor, comp_sorted_sim_items, total_items, device)
            comp_NEG_masked = mask_items(user_tensor, reverse_comp_sim_items, total_items, device)


            ##index of first item and second item by masking m3
            cmp_first_POS_index = get_index_in_the_list(comp_POS_masked, user_tensor, first_item_id, recommender, **kw_dict) + 1

            cmp_second_POS_index = get_index_in_the_list(comp_POS_masked, user_tensor, second_item_id, recommender, **kw_dict) + 1
            
            if (cmp_first_POS_index > cmp_second_POS_index):
                #print('index',i)
                return total_items, i
            

    
        return None, None

    # Find LXR masks
    comp_sim_items = find_LXR_mask(
        user_tensor, first_item_id, first_items_tensor, second_items_tensor, explainer
    )
    #### total items to be removed from the users profile to have reversion of ranking target item and contrastive item
    total_items, bin_index=process_sim_items(comp_sim_items, first_item_id, second_item_id, bins, user_tensor, user_hist_size, recommender, **kw_dict
    )
  
   
    return total_items, bin_index

In [ ]:

torch.manual_seed(42)
np.random.seed(42)

num_of_rand_users = 700 # number of users for evaluations
random_rows = np.random.choice(test_array.shape[0], num_of_rand_users, replace=False)
random_sampled_array = test_array[random_rows]

def lxr_training(trial):
    
   
    learning_rate = trial.suggest_float('learning_rate', 0.001, 0.01)
    alpha = trial.suggest_categorical('alpha',[0.5,1,10,50])
    gamma = trial.suggest_categorical('gamma',[0.5,1,10,50])
    lambda_neg =  trial.suggest_float('lambda_neg',0,30)
    lambda_pos =  trial.suggest_float('lambda_pos',0,30)
    batch_size = trial.suggest_categorical('batch_size', [8,16,32,64])
    explainer_hidden_size = trial.suggest_categorical('explainer_hidden_size', [32,64,128])
    epochs = 50
    patience = 10
    
    wandb.init(
        project=f"{data_name}_{recommender_name}_LXR_training",
        name=f"trial_{trial.number}",
        config={
        'learning_rate' : learning_rate,
        'alpha' : alpha,
        'lambda_neg' : lambda_neg,
        'lambda_pos' : lambda_pos,
        'batch_size' : batch_size,
        'explainer_hidden_size' : explainer_hidden_size,
        'architecture' : 'LXR_combined',
        'activation_function' : 'Tanh',
        'loss_type' : 'logloss',
        'optimize_for' : 'pos_at_20',
        'epochs':epochs
        })
    
    loader = torch.utils.data.DataLoader(train_array, batch_size=batch_size, shuffle=True)
    num_batches = int(np.ceil(train_array.shape[0] / batch_size))
    k=10

    num_of_bins = 10
    run_pos_at_20, run_neg_at_20 = [], []
    bin_indx_tot=[]
    flipping_comp_post=[]
    count_total=[]
    last_bin_indxs = []
    total_items_avg_tot= []
    total_items_list_tot= []
    train_losses = []
    count_mean=[]

    recommender.eval()

    num_features = num_items_dict[data_name]

    explainer = Explainer(num_features, num_items, explainer_hidden_size).to(device) 

    optimizer_comb = torch.optim.Adam(explainer.parameters(), learning_rate)
    


    loss_func = LXR_loss(lambda_pos, lambda_neg, alpha,gamma,k)

    print('======================== new run ========================')

    for epoch in range(epochs):
        if epoch%15 == 0 and epoch>0: # decrease learning rate every 15 epochs
            learning_rate*= 0.1
            optimizer_comb.lr = learning_rate

        

        train_loss = 0 
        total_pos_loss, total_neg_loss=0 , 0
        total_l1_loss=0

        explainer.train()
        for batch_index, samples in enumerate(loader):
            # prepare data for explainer:
            user_tensors = torch.Tensor(samples[:,:-1]).to(device)
            user_ids = samples[:,-1]
            ### target items batch for training explainer
            top1_item = np.array([top1_train[int(x)] for x in user_ids])

            ### contrastive items batch for training explainer
            top2_item = np.array([np.random.choice(top2_train[int(x)]) for x in user_ids])

        
            first_items_vectors = items_array[top1_item]
            second_items_vectors = items_array[top2_item]
            ### converting to a tensor
            first_items_tensors = torch.Tensor(first_items_vectors).to(device)
            second_items_tensors = torch.Tensor(second_items_vectors).to(device)
            n = user_tensors.shape[0]

            # zero grad:
            optimizer_comb.zero_grad()
            # forward:
            #### scoes for the target item
            #### scoes for the contrastive item
            #### comparative scoes
            expl_scores = explainer(user_tensors, first_items_tensors, second_items_tensors)

            # caclulate loss

            comb_loss, pos_loss, neg_loss, l1 = loss_func(user_tensors, first_items_tensors , top1_item, expl_scores,top2_item )

            train_loss += comb_loss*n
            total_pos_loss += pos_loss*n
            total_neg_loss += neg_loss*n
            total_l1_loss += l1*n

            # back propagation
            comb_loss.backward()
            optimizer_comb.step()

        train_metrics = {"train/train_loss": train_loss,
                         "train/pos_loss": total_pos_loss,
                         "train/neg_loss": total_neg_loss,
                         "train/l1_loss": total_l1_loss,
                         "train/epoch": epoch}

        torch.save(explainer.state_dict(), Path(checkpoints_path, f'LXR_{data_name}_{recommender_name}_{trial.number}_{epoch}_{explainer_hidden_size}_{lambda_pos}_{lambda_neg}.pt'))
        
        top_item_test_list=[]
        explainer.eval()
        #explainer_second.eval()
        bin_indxs_list=[]
        count=0
        total_items_list=[]
        for j in range(random_sampled_array.shape[0]):

            user_id = random_sampled_array[j][-1]
            user_tensor = torch.Tensor(random_sampled_array[j][:-1]).to(device)
            top1_item_test = top1_test[user_id]
            
            #top2_item_test = top2_test[user_id]

            top2_item_test=np.random.choice(top2_test[user_id])
            
            top_item_test_list.append((user_id, top1_item_test,top2_item_test))   

            first_item_vector = items_array[top1_item_test]
            second_item_vector = items_array[top2_item_test]

            first_items_tensor = torch.Tensor(first_item_vector).to(device)
            second_items_tensor = torch.Tensor(second_item_vector).to(device)

            total_items, bin_indx= calculate_pos_neg_k(user_tensor, top1_item_test, top2_item_test,first_items_tensor,second_items_tensor, num_of_bins, explainer, k=20)

            if total_items is not None:
                bin_indxs_list.append(bin_indx)
                count+=1
                total_items_list.append(total_items)
            
        
            

        count_total.append(count)  
    
        ## average of total items for each epoch
        total_items_avg=np.mean(total_items_list)
        ## creating a list for storing values of "total_items_avg" in each epoch
        total_items_avg_tot.append(total_items_avg)

        ### creating a list for storing values of "total_items_list" in each epoch
        total_items_list_tot.append(total_items_list)
        
        bin_indx_tot.append(bin_indxs_list)
        
    
        #def __init__(self, lambda_pos, lambda_neg,lambda_cmp1_pos, lambda_cmp2_pos, lambda_cmp_neg, alpha1, alpha2):

    


        print(f'Finished epoch {epoch} with  total_items_avg {total_items_avg} and count {count}')

   
        
    print(f'Stop at trial with learning rate {learning_rate}, batch size={batch_size}, explainer hidden size={explainer_hidden_size}, lambda_pos = {lambda_pos}, lambda_neg = {lambda_neg}, alpha_parameter = {alpha}, gamma_parameter ={gamma}. Best results at epoch {np.argmin(total_items_avg_tot)} with value {np.min(total_items_avg_tot)} and count at epoch {np.argmax(count_total)} with value {np.max(count_total)}')    
    
    
    
    return np.max(count_total) # return the best total items value in this trial

In [ ]:
logger = logging.getLogger()

logger.setLevel(logging.INFO)  # Setup the root logger.
logger.addHandler(logging.FileHandler(f"{data_name}_{recommender_name}_explainer_training.log", mode="w"))

optuna.logging.enable_propagation()  # Propagate logs to the root logger.
optuna.logging.disable_default_handler()  # Stop showing logs in sys.stderr.

study = optuna.create_study(direction='minimize')

logger.info("Start optimization.")
study.optimize(lxr_training, n_trials=1)

with open(f"{data_name}_{recommender_name}_explainer_training.log") as f:
    assert f.readline().startswith("A new study created")
    assert f.readline() == "Start optimization.\n"
    
    
# Print best hyperparameters and corresponding metric value
print("Best hyperparameters: {}".format(study.best_params))
print("Best metric value: {}".format(study.best_value))